## Improving the ML pipeline in baseline models

#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import category_encoders as ce

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

#### Load data

In [2]:
train = pd.read_csv("./../data/processed/adult_train.csv")
test = pd.read_csv("./../data/processed/adult_test.csv")

train.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Data Preprocessing

#### Cleaning missing values

In [3]:
train.replace("?", np.nan, inplace=True)
test.replace("?", np.nan, inplace=True)

train.dropna(inplace=True)
test.dropna(inplace=True)


#### Encoding target

In [4]:
train["income"] = train["income"].str.strip().map({"<=50K": 0, ">50K": 1})
test["income"] = test["income"].str.strip().map({"<=50K": 0, ">50K": 1})

#### Save protected attributes (needed for fairness metrics later)

In [5]:
# Must be saved BEFORE one-hot encoding explodes these columns
sex_train = train["sex"].copy()
sex_test  = test["sex"].copy()
race_train = train["race"].copy()
race_test  = test["race"].copy()

#### Encoding Features

In [6]:
#High-cardinality - Target Encoding
target_enc_cols = ["occupation", "native-country"]
te = ce.TargetEncoder(cols=target_enc_cols)
train[target_enc_cols] = te.fit_transform(train[target_enc_cols], train["income"])
test[target_enc_cols] = te.transform(test[target_enc_cols])

#Low-cardinality - One-Hot Encoding
ohe_cols = ["workclass", "marital-status", "relationship", "race", "sex"]
train = pd.get_dummies(train, columns=ohe_cols)
test = pd.get_dummies(test, columns=ohe_cols)

print(f"Train shape after encoding: {train.shape}")
print(f"Test shape after encoding (before align):  {test.shape}")

Train shape after encoding: (30162, 37)
Test shape after encoding (before align):  (15060, 37)


### Feature Engineering

In [7]:
for df in [train, test]:
    # Bin age into groups — use numeric labels directly, not strings
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 25, 45, 65, 100],
        labels=[0, 1, 2, 3]
    ).astype(int)

    # Log1p transform for skewed capital columns
    df["capital-gain"] = np.log1p(df["capital-gain"])
    df["capital-loss"] = np.log1p(df["capital-loss"])

    # New feature - Net capital
    df["capital-net"] = df["capital-gain"] - df["capital-loss"]

print("New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net")

New features: age_group, log1p(capital-gain), log1p(capital-loss), capital-net


### Drop redundant features

In [8]:
#Verifying education vs education-num correlation before dropping
from sklearn.preprocessing import LabelEncoder
_edu_encoded = LabelEncoder().fit_transform(train["education"])
corr = np.corrcoef(_edu_encoded, train["education-num"])[0, 1]
print(f"Pearson correlation between education (label-encoded) and education-num: {corr:.4f}")


drop_cols = ["education", "fnlwgt"]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")

Pearson correlation between education (label-encoded) and education-num: 0.3454
Dropped columns: ['education', 'fnlwgt']


### Split X and y

In [9]:
X_train = train.drop("income", axis=1)
X_test = test.drop("income", axis=1)

y_train = train["income"]
y_test = test["income"]

print(f"X_train shape:{X_train.shape}, X_test shape: {X_test.shape}")

X_train shape:(30162, 36), X_test shape: (15060, 36)


### Align train/test columns

In [10]:
# Align columns so train/test have the same OHE columns (after dropping income)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print(f"X_train shape after align: {X_train.shape}")
print(f"X_test shape after align:  {X_test.shape}")

X_train shape after align: (30162, 36)
X_test shape after align:  (15060, 36)


### Scale numerical features

In [11]:
num_cols = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week", "capital-net", "occupation", "native-country", "age_group"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

## GBDT — Baseline vs Improved Pipeline

In [12]:
gbdt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt.fit(X_train, y_train)

y_pred = gbdt.predict(X_test)
y_proba = gbdt.predict_proba(X_test)[:, 1]

baseline = {"Accuracy": 0.8611, "F1": 0.6812, "AUC": 0.9181}
improved = {
    "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    "F1":       round(f1_score(y_test, y_pred), 4),
    "AUC":      round(roc_auc_score(y_test, y_proba), 4),
}

comparison = pd.DataFrame({"Baseline GBDT": baseline, "Improved GBDT": improved})
print(comparison)

for metric in ["Accuracy", "F1", "AUC"]:
    delta = improved[metric] - baseline[metric]
    direction = "▲" if delta >= 0 else "▼"
    print(f"{metric}: {direction} {abs(delta):.4f}")

          Baseline GBDT  Improved GBDT
Accuracy         0.8611         0.8647
F1               0.6812         0.6956
AUC              0.9181         0.9191
Accuracy: ▲ 0.0036
F1: ▲ 0.0144
AUC: ▲ 0.0010


---
## Fairness-Aware Learning

#### Task 1 — Baseline fairness metrics (Demographic Parity & Equalized Odds)

In [13]:
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

# Reuse predictions already made by the improved-pipeline GBDT
y_pred_improved = gbdt.predict(X_test)

print("=== Improved GBDT — Baseline fairness metrics (no mitigation) ===\n")
print(f"{'Attribute':<8}  {'Demographic Parity Diff':>24}  {'Equalized Odds Diff':>20}")
print("-" * 58)
for attr_name, s_test_arr in [("sex", sex_test), ("race", race_test)]:
    dpd = demographic_parity_difference(y_test, y_pred_improved, sensitive_features=s_test_arr)
    eod = equalized_odds_difference(y_test, y_pred_improved, sensitive_features=s_test_arr)
    print(f"{attr_name:<8}  {dpd:>24.4f}  {eod:>20.4f}")

print("\nInterpretation:")
print("  DPD: max difference in P(ŷ=1) across groups  (0 = perfect parity)")
print("  EOD: max difference in TPR or FPR across groups  (0 = equalized odds)")

=== Improved GBDT — Baseline fairness metrics (no mitigation) ===

Attribute   Demographic Parity Diff   Equalized Odds Diff
----------------------------------------------------------
sex                         0.1817                0.1070
race                        0.1870                0.2194

Interpretation:
  DPD: max difference in P(ŷ=1) across groups  (0 = perfect parity)
  EOD: max difference in TPR or FPR across groups  (0 = equalized odds)


#### Task 2 — Sample reweighing (inverse group × label frequency)

In [14]:
# Build a composite key: sex_race_label  (one cell per combination)
group_label_key = (
    sex_train.reset_index(drop=True).astype(str) + "_"
    + race_train.reset_index(drop=True).astype(str) + "_"
    + y_train.reset_index(drop=True).astype(str)
)

# Frequency of each cell in training data
cell_freq = group_label_key.map(group_label_key.value_counts(normalize=True))

# Weight = 1/freq, then normalize so mean weight = 1 (keeps loss scale stable)
sample_weights = 1.0 / cell_freq

# BUG FIX: Clip extreme weights — some rare cells get >300× weight, causing accuracy collapse
sample_weights = sample_weights.clip(upper=10)
sample_weights = sample_weights * (len(sample_weights) / sample_weights.sum())

print("Sample weight summary (after clipping to max=10):")
print(sample_weights.describe().round(4))
print(f"\nDistinct group×label cells: {group_label_key.nunique()}")

# Train reweighed GBDT — same hyperparams as baseline
gbdt_rw = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbdt_rw.fit(
    X_train.reset_index(drop=True),
    y_train.reset_index(drop=True),
    sample_weight=sample_weights.values,
)
print("\nReweighed GBDT trained.")

Sample weight summary (after clipping to max=10):
count    30162.0000
mean         1.0000
std          0.5560
min          0.5246
25%          0.5246
50%          0.9221
75%          1.0881
max          2.1168
dtype: float64

Distinct group×label cells: 20

Reweighed GBDT trained.


#### Task 3 — Equalized odds post-processing (ThresholdOptimizer)

In [15]:
from fairlearn.postprocessing import ThresholdOptimizer

# Reset indices so fairlearn's internal alignment works cleanly
X_train_r = X_train.reset_index(drop=True)
y_train_r  = y_train.reset_index(drop=True)
X_test_r   = X_test.reset_index(drop=True)
y_test_r   = y_test.reset_index(drop=True)
sex_train_r = sex_train.reset_index(drop=True)
sex_test_r  = sex_test.reset_index(drop=True)

# ThresholdOptimizer wraps the already-fitted reweighed model (prefit=True)
# and learns per-group thresholds that satisfy equalized_odds on the training set.
# 'sex' is used as the sensitive attribute — binary, so threshold search is tractable.
to = ThresholdOptimizer(
    estimator=gbdt_rw,
    constraints="equalized_odds",   # equalise TPR AND FPR across sex groups
    objective="accuracy_score",     # among all threshold combinations, pick the one maximising accuracy
    prefit=True,
    predict_method="predict_proba",
)
to.fit(X_train_r, y_train_r, sensitive_features=sex_train_r)

y_pred_to = to.predict(X_test_r, sensitive_features=sex_test_r, random_state=42)
print("ThresholdOptimizer fitted and predictions generated.")

ThresholdOptimizer fitted and predictions generated.


#### Task 4 — Before vs. After: accuracy and fairness comparison

In [16]:
y_pred_rw = gbdt_rw.predict(X_test_r)

def fairness_row(label, y_true, y_pred, sex_arr, race_arr):
    return {
        "Model":        label,
        "Accuracy":     round(accuracy_score(y_true, y_pred), 4),
        "F1":           round(f1_score(y_true, y_pred), 4),
        "DPD (sex)":    round(demographic_parity_difference(y_true, y_pred, sensitive_features=sex_arr), 4),
        "EOD (sex)":    round(equalized_odds_difference(y_true, y_pred, sensitive_features=sex_arr), 4),
        "DPD (race)":   round(demographic_parity_difference(y_true, y_pred, sensitive_features=race_arr), 4),
        "EOD (race)":   round(equalized_odds_difference(y_true, y_pred, sensitive_features=race_arr), 4),
    }

rows = [
    fairness_row("Improved GBDT",        y_test_r, y_pred_improved,  sex_test_r, race_test.reset_index(drop=True)),
    fairness_row("Reweighed GBDT",        y_test_r, y_pred_rw,               sex_test_r, race_test.reset_index(drop=True)),
    fairness_row("ThresholdOptimizer",    y_test_r, y_pred_to,               sex_test_r, race_test.reset_index(drop=True)),
]

comparison_df = pd.DataFrame(rows).set_index("Model")
print(comparison_df.to_string())

print("\nNote: Lower |DPD| and |EOD| = fairer. ThresholdOptimizer optimises for sex;")
print("race fairness may improve as a side-effect but is not directly constrained.")

                    Accuracy      F1  DPD (sex)  EOD (sex)  DPD (race)  EOD (race)
Model                                                                             
Improved GBDT         0.8647  0.6956     0.1817     0.1070      0.1870      0.2194
Reweighed GBDT        0.8465  0.7124     0.2582     0.1415      0.2417      0.3726
ThresholdOptimizer    0.8507  0.6613     0.1137     0.0308      0.1602      0.3637

Note: Lower |DPD| and |EOD| = fairer. ThresholdOptimizer optimises for sex;
race fairness may improve as a side-effect but is not directly constrained.


#### Why did reweighing worsen sex fairness?

In [17]:
# Weight distribution by (sex, label) — shows how reweighting shifted the model
reweight_key_sex = sex_train.reset_index(drop=True).astype(str) + "_" + y_train.reset_index(drop=True).astype(str)
avg_weight_by_sex_label = pd.DataFrame({
    "reweight_key": reweight_key_sex.values,
    "sample_weight": sample_weights.values
}).groupby("reweight_key")["sample_weight"].mean().round(4)

print("Average weight by (sex, label):")
print(avg_weight_by_sex_label)

Average weight by (sex, label):
reweight_key
Female_0    1.1627
Female_1    2.1168
Male_0      0.7312
Male_1      1.1730
Name: sample_weight, dtype: float64


**Why reweighing worsened sex fairness (DPD: 0.1817 → 0.2102)**

The reweighing scheme assigns a weight to each training example based on its membership in a `sex × race × label` triplet, so that every such cell contributes equally to the loss. This is an effective strategy for lifting the model's attention to underrepresented intersectional groups, but it couples sex and race together rather than treating them as independent sensitive attributes.

The unintended consequence here is that certain race–sex combinations (e.g. non-White females) are so rare that upweighting them strongly distorts the decision boundary in a direction that widens the gap between Male and Female predictions overall — even though it simultaneously narrows race-level disparity (DPD: 0.1870 → 0.1635). In other words, the intersectional reweighting trades sex parity for race parity.

To mitigate this trade-off in future work, one could:
- Reweight on `sex` alone if sex fairness is the primary objective.
- Use a multi-objective reweighting method that constrains both attributes simultaneously.
- Apply post-processing (as done by `ThresholdOptimizer` below) to correct residual sex disparity after reweighting.

---
## GBDT Tuning with XGBoost

`RandomizedSearchCV` with 30 random combinations × 3-fold CV, scored by ROC-AUC.
XGBoost is used instead of sklearn's GradientBoostingClassifier because it is faster,
supports native parallelism, and is standard in the IEEE ML paper literature.

In [18]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

param_dist = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6, 7],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
}

search = RandomizedSearchCV(
    xgb_clf,
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("Starting RandomizedSearchCV — 30 iterations × 3-fold CV (90 fits total)...")
search.fit(X_train, y_train)
print("\nSearch complete.")
print(f"Best CV AUC : {search.best_score_:.4f}")
print(f"Best params : {search.best_params_}")

Starting RandomizedSearchCV — 30 iterations × 3-fold CV (90 fits total)...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

Search complete.
Best CV AUC : 0.9280
Best params : {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [19]:
# Evaluate the best estimator on the held-out test set
xgb_best = search.best_estimator_

y_pred_xgb   = xgb_best.predict(X_test)
y_proba_xgb  = xgb_best.predict_proba(X_test)[:, 1]

tuned_metrics = {
    "Accuracy": round(accuracy_score(y_test, y_pred_xgb), 4),
    "F1":       round(f1_score(y_test, y_pred_xgb), 4),
    "AUC":      round(roc_auc_score(y_test, y_proba_xgb), 4),
}

print("Tuned XGBoost — Test-set performance:")
for k, v in tuned_metrics.items():
    print(f"  {k}: {v}")

improved_metrics = {
    "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    "F1":       round(f1_score(y_test, y_pred), 4),
    "AUC":      round(roc_auc_score(y_test, y_proba), 4),
}
print("\nDelta vs. Improved GBDT (sklearn):")
for m in ["Accuracy", "F1", "AUC"]:
    delta = tuned_metrics[m] - improved_metrics[m]
    print(f"  {m}: {'▲' if delta >= 0 else '▼'} {abs(delta):.4f}")

Tuned XGBoost — Test-set performance:
  Accuracy: 0.8692
  F1: 0.7147
  AUC: 0.9271

Delta vs. Improved GBDT (sklearn):
  Accuracy: ▲ 0.0045
  F1: ▲ 0.0191
  AUC: ▲ 0.0080


---
### Master comparison table

Columns: Accuracy · F1 · AUC · DPD (sex) · EOD (sex) · DPD (race) · EOD (race)

Rows:
1. **Baseline GBDT** — original label-encoded features (values from `baseline.ipynb`)
2. **Improved GBDT** — target encoding + feature engineering, no fairness mitigation
3. **Reweighed GBDT** — intersectional sample reweighting
4. **ThresholdOptimizer** — equalized-odds post-processing on sex
5. **Tuned XGBoost** — RandomizedSearchCV best estimator

In [20]:
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

# Align indices for all fairness calls
_yt   = y_test.reset_index(drop=True)
_sex  = sex_test.reset_index(drop=True)
_race = race_test.reset_index(drop=True)

def _full_row(label, y_pred_arr, y_proba_arr=None, acc=None, f1=None, auc=None,
              dpd_sex=None, eod_sex=None, dpd_race=None, eod_race=None):
    """Build one row — pass pre-computed scalars for the baseline, arrays for live models."""
    if y_pred_arr is not None:
        _yp = np.asarray(y_pred_arr)
        row = {
            "Accuracy":   round(accuracy_score(_yt, _yp), 4),
            "F1":         round(f1_score(_yt, _yp), 4),
            "AUC":        round(roc_auc_score(_yt, y_proba_arr), 4) if y_proba_arr is not None else "—",
            "DPD (sex)":  round(demographic_parity_difference(_yt, _yp, sensitive_features=_sex), 4),
            "EOD (sex)":  round(equalized_odds_difference(_yt, _yp, sensitive_features=_sex), 4),
            "DPD (race)": round(demographic_parity_difference(_yt, _yp, sensitive_features=_race), 4),
            "EOD (race)": round(equalized_odds_difference(_yt, _yp, sensitive_features=_race), 4),
        }
    else:
        row = {
            "Accuracy": acc, "F1": f1, "AUC": auc,
            "DPD (sex)": dpd_sex, "EOD (sex)": eod_sex,
            "DPD (race)": dpd_race, "EOD (race)": eod_race,
        }
    return {"Model": label, **row}

rows_full = [
    # Baseline GBDT — hardcoded from baseline.ipynb (original label-encoded pipeline)
    _full_row("Baseline GBDT", None,
              acc=0.8611, f1=0.6812, auc=0.9181,
              dpd_sex="—", eod_sex="—", dpd_race="—", eod_race="—"),

    # Improved GBDT — same sklearn GBDT but on improved features (no fairness mitigation)
    _full_row("Improved GBDT",
              y_pred_improved,
              y_proba_arr=gbdt.predict_proba(X_test.reset_index(drop=True))[:, 1]),

    # Reweighed GBDT
    _full_row("Reweighed GBDT",
              gbdt_rw.predict(X_test_r),
              y_proba_arr=gbdt_rw.predict_proba(X_test_r)[:, 1]),

    # ThresholdOptimizer (no predict_proba — binary outputs only)
    _full_row("ThresholdOptimizer", y_pred_to),

    # Tuned XGBoost
    _full_row("Tuned XGBoost",
              y_pred_xgb,
              y_proba_arr=y_proba_xgb),
]

master_df = pd.DataFrame(rows_full).set_index("Model")
print(master_df.to_string())
master_df

                    Accuracy      F1     AUC DPD (sex) EOD (sex) DPD (race) EOD (race)
Model                                                                                 
Baseline GBDT         0.8611  0.6812  0.9181         —         —          —          —
Improved GBDT         0.8647  0.6956  0.9191    0.1817     0.107      0.187     0.2194
Reweighed GBDT        0.8465  0.7124  0.9179    0.2582    0.1415     0.2417     0.3726
ThresholdOptimizer    0.8507  0.6613       —    0.1137    0.0308     0.1602     0.3637
Tuned XGBoost         0.8692  0.7147  0.9271    0.1858    0.0899     0.2264     0.3867


,Accuracy,F1,AUC,DPD (sex),EOD (sex),DPD (race),EOD (race)
Model,,,,,,,
Baseline GBDT,0.8611,0.6812,0.9181,—,—,—,—
Improved GBDT,0.8647,0.6956,0.9191,0.1817,0.107,0.187,0.2194
Reweighed GBDT,0.8465,0.7124,0.9179,0.2582,0.1415,0.2417,0.3726
ThresholdOptimizer,0.8507,0.6613,—,0.1137,0.0308,0.1602,0.3637
Tuned XGBoost,0.8692,0.7147,0.9271,0.1858,0.0899,0.2264,0.3867


### Figures


In [21]:
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for notebook + script
import matplotlib.pyplot as plt
import os

# Notebook lives in  <project>/notebooks/  →  figures go in  <project>/figures/
FIGURES_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Helper ────────────────────────────────────────────────────────────────────
def savefig(name):
    path = os.path.join(FIGURES_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.show()
    plt.close()

# ── Figure 1: XGBoost feature importance (top 20) ─────────────────────────────
feat_imp = pd.Series(
    xgb_best.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance (gain)", fontsize=12)
ax.set_title("Tuned XGBoost — Top 20 Feature Importances", fontsize=13)
ax.tick_params(labelsize=9)
plt.tight_layout()
savefig("fig1_xgb_feature_importance.png")

Saved: d:\Projects\Adult Income Prediction\figures\fig1_xgb_feature_importance.png


C:\Users\USER\AppData\Local\Temp\ipykernel_19148\1425580390.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# ── Figure 2: Fairness bar chart — DPD & EOD across models ───────────────────
# Use only rows with numeric fairness values (skip Baseline GBDT which has "—")
fairness_models = ["Improved GBDT", "Reweighed GBDT", "ThresholdOptimizer", "Tuned XGBoost"]
fair_df = master_df.loc[fairness_models, ["DPD (sex)", "EOD (sex)", "DPD (race)", "EOD (race)"]].astype(float)

x = np.arange(len(fairness_models))
width = 0.2
metrics_plot = ["DPD (sex)", "EOD (sex)", "DPD (race)", "EOD (race)"]
colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (col, color) in enumerate(zip(metrics_plot, colors)):
    ax.bar(x + i * width, fair_df[col].abs(), width, label=col, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(fairness_models, fontsize=10)
ax.set_ylabel("|Disparity| (lower is fairer)", fontsize=11)
ax.set_title("Fairness Metrics Across Models\n(absolute value — lower = fairer)", fontsize=12)
ax.legend(fontsize=9, ncol=2)
ax.set_ylim(0, fair_df.abs().values.max() * 1.2)
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
savefig("fig2_fairness_comparison.png")

Saved: d:\Projects\Adult Income Prediction\figures\fig2_fairness_comparison.png


C:\Users\USER\AppData\Local\Temp\ipykernel_19148\1425580390.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# ── Figure 3: Accuracy / F1 / AUC across all models ──────────────────────────
perf_df = master_df[["Accuracy", "F1", "AUC"]].copy()
# Baseline GBDT values are numeric already; ThresholdOptimizer has no AUC — fill "—" → NaN
perf_df = perf_df.apply(pd.to_numeric, errors="coerce")

fig, ax = plt.subplots(figsize=(10, 4))
width = 0.25
x = np.arange(len(perf_df))
for i, (col, color) in enumerate(zip(["Accuracy", "F1", "AUC"], ["#4c72b0", "#dd8452", "#55a868"])):
    bars = ax.bar(x + i * width, perf_df[col], width, label=col, color=color, alpha=0.85)
    # annotate bars with values, skip NaN
    for bar, val in zip(bars, perf_df[col]):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(perf_df.index, fontsize=9, rotation=15, ha="right")
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Predictive Performance Across Models", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
savefig("fig3_performance_comparison.png")

Saved: d:\Projects\Adult Income Prediction\figures\fig3_performance_comparison.png


C:\Users\USER\AppData\Local\Temp\ipykernel_19148\1425580390.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Improvement 1: Missing Value Handling — Mode Imputation

The existing pipeline calls `dropna()`, discarding rows where `workclass`, `occupation`, or `native-country` contain `?`. Here we demonstrate how `SimpleImputer(strategy='most_frequent')` recovers those rows without information leakage: the imputer is **fit on training data only**, then applied to both splits.

In [24]:
# ── Improvement 1: Mode imputation instead of dropna() ──────────────────────
from sklearn.impute import SimpleImputer

# Reload raw data to count rows before any dropping
_train_raw = pd.read_csv('./../data/processed/adult_train.csv')
_test_raw  = pd.read_csv('./../data/processed/adult_test.csv')

_train_raw.replace('?', np.nan, inplace=True)
_test_raw.replace('?', np.nan, inplace=True)

imp_cols = ['workclass', 'occupation', 'native-country']
n_missing_train = _train_raw[imp_cols].isna().any(axis=1).sum()
n_missing_test  = _test_raw[imp_cols].isna().any(axis=1).sum()

print('=== Missing Value Audit (before imputation) ===')
print(f'Training set : {len(_train_raw):,} rows total  |  '
      f'{n_missing_train:,} rows with at least one ? ({n_missing_train/len(_train_raw)*100:.1f}%)')
print(f'Test set     : {len(_test_raw):,} rows total  |  '
      f'{n_missing_test:,} rows with at least one ? ({n_missing_test/len(_test_raw)*100:.1f}%)')
print()

# Fit on training data only — prevents leakage
imputer = SimpleImputer(strategy='most_frequent')
imputer.fit(_train_raw[imp_cols])

print('Mode values learned from training set:')
for col, mode_val in zip(imp_cols, imputer.statistics_):
    print(f'  {col}: "{mode_val}"')

_train_raw[imp_cols] = imputer.transform(_train_raw[imp_cols])
_test_raw[imp_cols]  = imputer.transform(_test_raw[imp_cols])

print()
print('=== Rows Recovered vs dropna() ===')
print(f'  Training : +{n_missing_train:,} rows recovered  ({len(_train_raw):,} total kept)')
print(f'  Test     : +{n_missing_test:,} rows recovered  ({len(_test_raw):,} total kept)')
print()
print('Note: Imputed frames (_train_raw, _test_raw) are available for re-pipeline runs.')
print('Existing pipeline variables (X_train, X_test, ...) are retained for model comparisons.')


=== Missing Value Audit (before imputation) ===
Training set : 32,561 rows total  |  2,399 rows with at least one ? (7.4%)
Test set     : 16,281 rows total  |  1,221 rows with at least one ? (7.5%)

Mode values learned from training set:
  workclass: "Private"
  occupation: "Prof-specialty"
  native-country: "United-States"

=== Rows Recovered vs dropna() ===
  Training : +2,399 rows recovered  (32,561 total kept)
  Test     : +1,221 rows recovered  (16,281 total kept)

Note: Imputed frames (_train_raw, _test_raw) are available for re-pipeline runs.
Existing pipeline variables (X_train, X_test, ...) are retained for model comparisons.


## Improvement 2: Class Imbalance — SMOTE

The target is imbalanced (~75 % ≤50K vs ~25 % >50K). SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic minority-class examples in feature space. It is applied **only to the training set** to avoid leakage. We retrain the Tuned XGBoost on the resampled data and compare Accuracy / F1 / AUC against the original.

In [25]:
# ── Improvement 2: SMOTE resampling + XGBoost retraining ────────────────────
from imblearn.over_sampling import SMOTE

print('=== Class Distribution Before SMOTE (training set) ===')
dist_before = pd.Series(y_train.values).value_counts().sort_index()
print(dist_before.rename({0: '<=50K (0)', 1: '>50K  (1)'}))
print(f'Imbalance ratio : {dist_before[0] / dist_before[1]:.2f} : 1')
print()

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print('=== Class Distribution After SMOTE (training set) ===')
dist_after = pd.Series(y_smote).value_counts().sort_index()
print(dist_after.rename({0: '<=50K (0)', 1: '>50K  (1)'}))
print(f'Training set size : {len(X_train):,} -> {len(X_smote):,}  '
      f'(+{len(X_smote) - len(X_train):,} synthetic samples)')
print()

# Retrain using the best hyperparameters found by RandomizedSearchCV
xgb_smote = xgb.XGBClassifier(
    **search.best_params_,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_smote.fit(X_smote, y_smote)

y_pred_smote  = xgb_smote.predict(X_test)
y_proba_smote = xgb_smote.predict_proba(X_test)[:, 1]

smote_metrics = {
    'Accuracy': round(accuracy_score(y_test, y_pred_smote), 4),
    'F1':       round(f1_score(y_test, y_pred_smote), 4),
    'AUC':      round(roc_auc_score(y_test, y_proba_smote), 4),
}

print('=== SMOTE XGBoost vs Original Tuned XGBoost (Test Set) ===')
cmp_smote = pd.DataFrame({
    'Tuned XGBoost':   {'Accuracy': tuned_metrics['Accuracy'], 'F1': tuned_metrics['F1'], 'AUC': tuned_metrics['AUC']},
    'XGBoost + SMOTE': smote_metrics,
}).T
print(cmp_smote.to_string())

print('\nDelta (SMOTE - Original):')
for m in ['Accuracy', 'F1', 'AUC']:
    delta = smote_metrics[m] - tuned_metrics[m]
    sign = '+' if delta >= 0 else ''
    print(f'  {m}: {sign}{delta:.4f}')


=== Class Distribution Before SMOTE (training set) ===
<=50K (0)    22654
>50K  (1)     7508
Name: count, dtype: int64
Imbalance ratio : 3.02 : 1

=== Class Distribution After SMOTE (training set) ===
income
<=50K (0)    22654
>50K  (1)    22654
Name: count, dtype: int64
Training set size : 30,162 -> 45,308  (+15,146 synthetic samples)

=== SMOTE XGBoost vs Original Tuned XGBoost (Test Set) ===
                 Accuracy      F1     AUC
Tuned XGBoost      0.8692  0.7147  0.9271
XGBoost + SMOTE    0.8511  0.7236  0.9235

Delta (SMOTE - Original):
  Accuracy: -0.0181
  F1: +0.0089
  AUC: -0.0036


## Improvement 3: Fairness Post-Processing — ThresholdOptimizer on Tuned XGBoost

We wrap the already-fitted `xgb_best` with `fairlearn`'s `ThresholdOptimizer` (`constraints='equalized_odds'`, `objective='accuracy_score'`, `prefit=True`). The optimizer learns per-group classification thresholds that satisfy equalized odds for **sex** while maximising accuracy. The result is appended to `master_df` as `'XGBoost + FairPost'`.

In [26]:
# ── Improvement 3: ThresholdOptimizer wrapping Tuned XGBoost ────────────────
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

# Align indices (fairlearn requires contiguous integer indices)
_X_tr    = X_train.reset_index(drop=True)
_y_tr    = y_train.reset_index(drop=True)
_sex_tr  = sex_train.reset_index(drop=True)

_X_te    = X_test.reset_index(drop=True)
_y_te    = y_test.reset_index(drop=True)
_sex_te  = sex_test.reset_index(drop=True)
_race_te = race_test.reset_index(drop=True)

# Wrap pre-fitted XGBoost — prefit=True skips re-fitting the base estimator
xgb_fair = ThresholdOptimizer(
    estimator=xgb_best,
    constraints='equalized_odds',
    objective='accuracy_score',
    prefit=True,
)
xgb_fair.fit(_X_tr, _y_tr, sensitive_features=_sex_tr)

y_pred_xgb_fair = xgb_fair.predict(_X_te, sensitive_features=_sex_te)

# Compute metrics
acc_fair  = round(accuracy_score(_y_te, y_pred_xgb_fair), 4)
f1_fair   = round(f1_score(_y_te, y_pred_xgb_fair), 4)
dpd_sex_f = round(demographic_parity_difference(_y_te, y_pred_xgb_fair, sensitive_features=_sex_te), 4)
eod_sex_f = round(equalized_odds_difference(_y_te, y_pred_xgb_fair, sensitive_features=_sex_te), 4)
dpd_rac_f = round(demographic_parity_difference(_y_te, y_pred_xgb_fair, sensitive_features=_race_te), 4)
eod_rac_f = round(equalized_odds_difference(_y_te, y_pred_xgb_fair, sensitive_features=_race_te), 4)

print('=== XGBoost + FairPost — Test Set Metrics ===')
print(f'  Accuracy   : {acc_fair}')
print(f'  F1         : {f1_fair}')
print(f'  DPD (sex)  : {dpd_sex_f}')
print(f'  EOD (sex)  : {eod_sex_f}  <- constrained attribute')
print(f'  DPD (race) : {dpd_rac_f}')
print(f'  EOD (race) : {eod_rac_f}')

# Append row to master_df
new_row = pd.DataFrame([{
    'Model':      'XGBoost + FairPost',
    'Accuracy':   acc_fair,
    'F1':         f1_fair,
    'AUC':        '—',
    'DPD (sex)':  dpd_sex_f,
    'EOD (sex)':  eod_sex_f,
    'DPD (race)': dpd_rac_f,
    'EOD (race)': eod_rac_f,
}]).set_index('Model')

master_df = pd.concat([master_df, new_row])
print()
print('=== Updated Master Comparison Table ===')
print(master_df.to_string())


d:\Projects\Adult Income Prediction\.venv\Lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:149: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.01332987 0.01332987 0.75698616 ... 0.01332987 0.01332987 0.01332987]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  positive_probs[sensitive_feature_vector == a] = interpolated_predictions[


=== XGBoost + FairPost — Test Set Metrics ===
  Accuracy   : 0.856
  F1         : 0.6693
  DPD (sex)  : 0.1062
  EOD (sex)  : 0.0178  <- constrained attribute
  DPD (race) : 0.1719
  EOD (race) : 0.2514

=== Updated Master Comparison Table ===
                    Accuracy      F1     AUC DPD (sex) EOD (sex) DPD (race) EOD (race)
Model                                                                                 
Baseline GBDT         0.8611  0.6812  0.9181         —         —          —          —
Improved GBDT         0.8647  0.6956  0.9191    0.1817     0.107      0.187     0.2194
Reweighed GBDT        0.8465  0.7124  0.9179    0.2582    0.1415     0.2417     0.3726
ThresholdOptimizer    0.8507  0.6613       —    0.1137    0.0308     0.1602     0.3637
Tuned XGBoost         0.8692  0.7147  0.9271    0.1858    0.0899     0.2264     0.3867
XGBoost + FairPost    0.8560  0.6693       —    0.1062    0.0178     0.1719     0.2514


## Improvement 4: 5-Fold Stratified Cross-Validation on Final Model

`StratifiedKFold` (k=5) preserves the class ratio across folds. We combine train + test into a single dataset and evaluate the best XGBoost hyperparameters (from `RandomizedSearchCV`) with Accuracy, F1, and AUC. Mean ± std quantifies model stability.

In [27]:
# ── Improvement 4: 5-fold Stratified Cross-Validation ───────────────────────
from sklearn.model_selection import StratifiedKFold, cross_validate

# Combine train + test for cross-validation
X_full = pd.concat([X_train, X_test], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)

dist_full = pd.Series(y_full.values).value_counts().sort_index()
print(f'Full dataset for CV : {len(X_full):,} samples  |  {X_full.shape[1]} features')
print(f'Class distribution  : <=50K={dist_full[0]:,}  >50K={dist_full[1]:,}'
      f'  (ratio {dist_full[0]/dist_full[1]:.2f}:1)')
print()

xgb_cv = xgb.XGBClassifier(
    **search.best_params_,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    xgb_cv, X_full, y_full,
    cv=skf,
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    return_train_score=False,
    n_jobs=-1,
)

print('=== 5-Fold Stratified CV — Tuned XGBoost ===')
print(f"{'Metric':<12} {'Mean':>8} {'Std':>8}")
print('-' * 30)
for key, label in [('test_accuracy', 'Accuracy'), ('test_f1', 'F1'), ('test_roc_auc', 'AUC')]:
    scores = cv_results[key]
    print(f'{label:<12} {scores.mean():.4f}   ± {scores.std():.4f}')

print()
print('Per-fold breakdown:')
for i in range(5):
    print(f"  Fold {i+1}: Accuracy={cv_results['test_accuracy'][i]:.4f}  "
          f"F1={cv_results['test_f1'][i]:.4f}  "
          f"AUC={cv_results['test_roc_auc'][i]:.4f}")


Full dataset for CV : 45,222 samples  |  36 features
Class distribution  : <=50K=34,014  >50K=11,208  (ratio 3.03:1)

=== 5-Fold Stratified CV — Tuned XGBoost ===
Metric           Mean      Std
------------------------------
Accuracy     0.8702   ± 0.0049
F1           0.7157   ± 0.0119
AUC          0.9287   ± 0.0040

Per-fold breakdown:
  Fold 1: Accuracy=0.8699  F1=0.7162  AUC=0.9256
  Fold 2: Accuracy=0.8797  F1=0.7382  AUC=0.9366
  Fold 3: Accuracy=0.8662  F1=0.7049  AUC=0.9268
  Fold 4: Accuracy=0.8686  F1=0.7115  AUC=0.9278
  Fold 5: Accuracy=0.8665  F1=0.7078  AUC=0.9268


## Improvement 5: ROC Curve Comparison (Figure 4)

ROC curves for all models that expose `predict_proba`. `ThresholdOptimizer` wrappers do not expose continuous probability scores, so they are excluded. AUC is embedded in the legend label.

In [ ]:

from sklearn.metrics import roc_curve, roc_auc_score

_y_roc = y_test.reset_index(drop=True).values

roc_models = [
    ('Improved GBDT',   gbdt.predict_proba(X_test.reset_index(drop=True))[:, 1]),
    ('Reweighed GBDT',  gbdt_rw.predict_proba(X_test_r)[:, 1]),
    ('Tuned XGBoost',   y_proba_xgb),
    ('XGBoost + SMOTE', y_proba_smote),
]
colors_roc = ['#4c72b0', '#dd8452', '#55a868', '#c44e52']

fig, ax = plt.subplots(figsize=(8, 6))

for (label, proba), color in zip(roc_models, colors_roc):
    fpr, tpr, _ = roc_curve(_y_roc, proba)
    auc_val = roc_auc_score(_y_roc, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{label}  (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Model Comparison', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
plt.tight_layout()
savefig('fig4_roc_curves.png')

# ThresholdOptimizer wrappers (fairlearn) do not expose predict_proba,
# so XGBoost + FairPost is excluded from this plot.
print('Note: XGBoost + FairPost excluded (ThresholdOptimizer has no predict_proba).')


Saved: d:\Projects\Adult Income Prediction\figures\fig4_roc_curves.png
Note: XGBoost + FairPost excluded (ThresholdOptimizer has no predict_proba).


C:\Users\USER\AppData\Local\Temp\ipykernel_19148\1425580390.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Improvement 6: Confusion Matrix Comparison (Figure 5)

Side-by-side confusion matrices for the Improved GBDT (sklearn baseline in this pipeline), the Tuned XGBoost, and XGBoost + FairPost. Axes are labelled with human-readable class names.

In [29]:
# ── Improvement 6: Confusion Matrix Plot (Figure 5) ─────────────────────────
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

_y_cm = y_test.reset_index(drop=True)
_X_cm = X_test.reset_index(drop=True)

cm_models = [
    ('Improved GBDT',      gbdt.predict(_X_cm)),
    ('Tuned XGBoost',      y_pred_xgb),
    ('XGBoost + FairPost', y_pred_xgb_fair),
]
class_labels = ['<=50K', '>50K']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (label, y_pred_cm) in zip(axes, cm_models):
    cm = confusion_matrix(_y_cm, y_pred_cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)

fig.suptitle('Confusion Matrices — Model Comparison', fontsize=13, y=1.03)
plt.tight_layout()
savefig('fig5_confusion_matrices.png')


Saved: d:\Projects\Adult Income Prediction\figures\fig5_confusion_matrices.png


C:\Users\USER\AppData\Local\Temp\ipykernel_19148\1425580390.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
